In [1]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow
import unicodedata



# Carregando variáveis de ambiente
load_dotenv()

True

In [2]:
# Variáveis de ambiente para autenticação e configuração do BigQuery
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
project_id = os.getenv("PROJECT_ID")

# Tabela SILVER no BigQuery
tabela_silver = os.getenv("SILVER_TABLE")

# Tabela GOLD no BigQuery
# Dim Calendario
table_gold_produtos = os.getenv("GOLD_PRODUTOS")

In [3]:
# Inicializa o cliente do BigQuery usando credenciais de serviço
# O parâmetro 'project_id' especifica o projeto GCP onde as queries serão executadas
client = bigquery.Client.from_service_account_json(credencial, project=project_id)


query = f"""
SELECT *
FROM `{tabela_silver}`
"""

In [4]:
# Executa e converte pra DataFrame
resultado = client.query(query)
df = resultado.to_dataframe()

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [5]:
df_produto = df[["produto"]] \
    .drop_duplicates(subset="produto") \
    .reset_index(drop=True)

df_produto.insert(0, "produto_sk", range(1, len(df_produto) + 1))

In [7]:
# Criação e carga da tabela Gold

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Gold seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_produto,
    table_gold_produtos,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Gold Produto criada e dados carregados com sucesso")

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Tabela Gold Produto criada e dados carregados com sucesso
